# Understanding Transformers Through Complexity and Cost
## A Ladder of Architectures for English and Tagalog/Taglish Sarcasm Detection
**Machine Learning 3 — Mini Project 2 — MSDS 2026**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.viz import plot_complexity_ladder, plot_transfer_collapse

RESULTS = config.RESULTS_DIR
results_df = pd.read_csv(f"{RESULTS}/results_full.csv")
summary_df = pd.read_csv(f"{RESULTS}/summary_df.csv")
stats_df   = pd.read_csv(f"{RESULTS}/stats_df.csv")

TRAINED   = summary_df[summary_df["model_family"] != "llm_reference"]
REFERENCE = summary_df[summary_df["model_family"] == "llm_reference"]

## 1 Executive Summary

We study sarcasm detection as a binary classification task across English
and Tagalog/Taglish, using it as a lens to examine the trade-off between
architectural complexity and predictive payoff. We evaluate a ladder of
models — a TF-IDF baseline, TextCNN, BiGRU, from-scratch single-head,
multi-head, and stacked Transformer-encoder classifiers, and a fine-tuned
multilingual DistilBERT/mBERT/XLM-R — alongside a zero-shot LLM reference.

Our central finding is that, on this small dataset ({{N_TRAIN}} training
examples), **added architectural complexity does not reliably improve
performance**. The TF-IDF baseline reaches {{TFIDF_F1}} macro-F1 in EN→EN,
matching or exceeding every from-scratch and fine-tuned neural model.
From-scratch attention collapses sharply under cross-lingual transfer
(EN→TL), while target-language fine-tuning (TL→TL) recovers much of that
loss. The zero-shot LLM reference ({{LLM_TL_F1}} on TL) outperforms all
trained models, underscoring that capability here stems from large-scale
pretraining rather than from architecture. We recommend, for low-resource
Taglish sarcasm detection, a strong classical baseline or a pretrained
multilingual model over bespoke architectures, with an LLM as a
cost-bearing upper reference.

## 2 Introduction

Sarcasm detection is difficult because the intended meaning of an
utterance contradicts its literal wording, and the cue is often carried
by surrounding conversational context. We formulate the task as binary
classification: given a target utterance with its preceding context,
predict *sarcastic* or *not sarcastic*.

Beyond raw accuracy, we ask a methodological question: as we climb a
ladder of increasingly complex architectures, does each rung earn its
added parameters and compute? We recapitulate the historical progression
taught in the course — classical features, CNNs, RNNs, attention, the
Transformer — as a controlled experiment on one task, then extend it to a
zero-shot LLM reference.

**Objective.** Quantify the complexity–payoff trade-off for sarcasm
detection, and measure how well models transfer between English and
Tagalog/Taglish.

**Scope and limitations.** The dataset is small ({{N_TOTAL}} aligned
examples per language) and text-only, whereas the original MUStARD is
multimodal. Results should be read as evidence about the small-data,
text-only regime, not about sarcasm detection in general.

## 3 Business Value

Filipino/Taglish is a low-resource language for NLP yet a dominant
language of Philippine social media and customer interaction. Sarcasm
routinely defeats naive sentiment and moderation filters: a sarcastic
complaint is scored as positive, sarcastic hostility evades detection.

This work addresses a concrete build-vs-buy decision for any organisation
doing Philippine social listening or content moderation: *can an
English-trained model be reused for Taglish, must it be fine-tuned on
local data, or is a pay-per-call LLM the better option?* Our EN→TL versus
TL→TL comparison answers the first two, and the complexity ladder shows
how much model — and compute budget — is actually needed to clear a useful
accuracy bar. For cost-constrained deployment, a finding that a cheap
TF-IDF model is competitive has direct budget implications.

## 4 Data

**Source.** The English data is the text-only MUStARD sarcasm dataset;
each item has a conversational context, a target utterance, and a binary
sarcasm label. The Tagalog/Taglish set is an aligned translation of the
same items, carrying both its own label (`sarcasm_tl`) and the original
English label, which lets us measure label shift.

**Translation provenance.** The Taglish data was produced by
machine-translating MUStARD, then validating with **three human raters**,
judging for naturalness, and re-translating toward more natural Taglish
where needed. Inter-rater agreement (Fleiss' κ = {{KAPPA}}) and the
label-shift rate ({{SHIFT_PCT}}% of TL labels differ from the English
original) quantify how much sarcasm survives translation.

**Size and split.** {{N_TOTAL}} aligned examples per language;
approximately {{N_TRAIN}} train / {{N_TEST}} test under a 70/30 split.
The split is performed on the original conversation id rather than on
rows, preventing the same item from leaking across train and test in
different languages.

**Preprocessing.** Missing context/utterance fields are filled as empty
strings; inputs are constructed as `[CONTEXT] ... [UTTERANCE] ...`.

In [ ]:
from src.data import load_all_data
from src.eda import run_eda, compute_fleiss_kappa
df_all = load_all_data()
label_shift_df = run_eda(df_all)

## 5 Methodology

We adopt a **two-act design: optimise first, then test generalizability.**

**Act 1 (Optimise, EN→EN).** We hold the setting fixed and climb the
architecture ladder, tuning each model and recording macro-F1, parameter
count, and training time:

1. TF-IDF + Logistic Regression / Linear SVM (classical floor)
2. TextCNN (convolutional)
3. BiGRU (recurrent)
4. Single-head self-attention (QKV implemented from scratch)
5. Multi-head attention
6. N-layer Transformer encoder (positional encoding; depths 1/2/4)
7. Pretrained multilingual Transformer (DistilBERT / mBERT / XLM-R)

A **majority-class baseline** anchors the chance floor (~50%). Ablations
on the encoder — number of heads, number of layers, sinusoidal vs.
learnable vs. no positional encoding — are run in EN→EN only.

**Regularization.** Neural models use dropout, weight decay, a carved
validation split, and early stopping on validation macro-F1.

**Act 2 (Generalize).** The tuned models are evaluated across all three
settings — EN→EN, EN→TL, TL→TL — over {{N_SEEDS}} random seeds, reported
as mean ± std, with paired Wilcoxon tests for transfer cost and
fine-tuning benefit.

**Reference tier.** A zero-shot LLM ({{LLM_MODEL}}) judges each test
utterance in-language. It is reported separately because it is not trained
on our split and is not a level-field competitor.

**Evaluation.** Primary metric is macro-F1 (both classes matter), with
accuracy and sarcasm-class F1 reported alongside.

## 6 Results and Discussion

We organize results by research question. Trained models are reported
separately from the LLM reference tier throughout, since the latter is
not trained on our split.

### 6.1 RQ3 — Complexity vs. Payoff

**Figure 1** (above) shows EN→EN macro-F1 against models ordered by
parameter count. The result contradicts the intuition that more
complexity helps: the **TF-IDF baseline ({{TFIDF_F1}}) is the strongest
trained model in EN→EN**, matching or exceeding both the from-scratch
attention rungs (single-head {{SH_F1}}, multi-head {{MH_F1}}, encoder
{{ENC_F1}}) and the fine-tuned multilingual Transformers ({{MBERT_F1}}–
{{XLMR_F1}}), despite the latter having three to four orders of magnitude
more parameters.

Two conclusions follow, both deliberately cautious. First, **from-scratch
attention does not beat the bag-of-words floor** on {{N_TRAIN}} examples —
there is too little data to learn useful attention patterns from scratch.
Second, contrary to our original hypothesis, **pretraining does not produce
a decisive advantage** here either: the best pretrained model is close to,
but does not clearly exceed, TF-IDF in EN→EN. The payoff for complexity, in
this small text-only regime, is essentially flat.

The ablation runs (heads ∈ {1,2,4}; layers ∈ {1,2,4}; positional encoding
sinusoidal/learnable/none) likewise show {{ABLATION_SUMMARY}}, reinforcing
that capacity is not the binding constraint — data is.

In [ ]:
# RQ1: EN->EN vs EN->TL for trained models
rq1 = (TRAINED[TRAINED["setting"].isin(["EN->EN", "EN->TL"])]
       .sort_values(["model_name", "setting"]))
display(rq1)

fig, ax = plot_transfer_collapse(TRAINED)
plt.show()

### 6.2 RQ1 — Cross-Lingual Transfer

**Figure 2** contrasts EN→EN and EN→TL macro-F1. Transfer weakens
performance for every trained model, but **not uniformly**, and the
pattern is the substantive finding:

- **Scratch neural models collapse.** Single-head ({{SH_DELTA}}),
  multi-head ({{MH_DELTA}}), the encoder ({{ENC_DELTA}}), TextCNN
  ({{CNN_DELTA}}), and BiGRU ({{BIGRU_DELTA}}) drop sharply toward chance.
  Their vocabularies are English-only, so Tagalog test tokens are largely
  unseen — they structurally cannot transfer.
- **TF-IDF does not collapse.** It drops modestly ({{TFIDF_DELTA}}) and
  remains one of the better EN→TL models, because shared tokens (English
  loanwords, names, code-switched fragments common in Taglish) still carry
  signal.
- **Pretrained multilingual models drop least**, retaining the most
  performance thanks to a shared subword vocabulary.

We therefore **reject the strong form of our original hypothesis** ("only
pretrained models transfer"): TF-IDF transfers competitively. The accurate
claim is that *from-scratch neural models* collapse, while classical and
pretrained multilingual models degrade gracefully. All scratch-model drops
are statistically significant (Wilcoxon p ≈ {{RQ1_P}}).

In [ ]:
# RQ2: EN->TL vs TL->TL for trained models
rq2 = (TRAINED[TRAINED["setting"].isin(["EN->TL", "TL->TL"])]
       .sort_values(["model_name", "setting"]))
display(rq2)

# paired significance (transfer cost and fine-tuning benefit)
display(stats_df)

### 6.3 RQ2 — Target-Language Fine-Tuning

Training directly on Taglish (TL→TL) versus transferring from English
(EN→TL) helps **most** models, but the evidence is not uniform:

- **Scratch and lower models improve clearly and significantly.** BiGRU
  ({{BIGRU_FT}}, p={{BIGRU_FT_P}}), TextCNN ({{CNN_FT}}, p={{CNN_FT_P}}),
  single-head ({{SH_FT}}, p={{SH_FT_P}}), and the encoder ({{ENC_FT}},
  p={{ENC_FT_P}}) all recover much of their transfer loss.
- **Pretrained models improve but with weaker evidence.** With
  {{N_EXP_SEEDS}} seeds, DistilBERT gains {{DBERT_FT}} (p={{DBERT_FT_P}})
  and mBERT gains {{MBERT_FT}} (p={{MBERT_FT_P}}).
- **XLM-R is the exception.** It performs *worse* on TL→TL
  ({{XLMR_FT}}, p={{XLMR_FT_P}}) and shows high seed-to-seed variance,
  contradicting the simple "target-language training always helps" claim.

The honest conclusion: **target-language fine-tuning generally helps, but
is not universal**, and we explicitly flag XLM-R's instability rather than
average it away.

In [ ]:
# RQ4: LLM reference vs trained ceiling, in-language
display(REFERENCE.sort_values("setting"))
# Best trained model per setting, for side-by-side context
best_trained = (TRAINED.sort_values("mean_macro_f1", ascending=False)
                .groupby("setting").head(1))
display(best_trained[["setting", "model_name", "mean_macro_f1"]])

### 6.4 RQ4 — Reference Ceilings

**LLM reference.** A zero-shot {{LLM_MODEL}} judge reaches {{LLM_EN_F1}}
macro-F1 in English and {{LLM_TL_F1}} in Taglish — **above every trained
model in both settings**. Notably it is strongest on Taglish, suggesting
its multilingual pretraining handles code-switching better than anything
trainable on {{N_TRAIN}} examples. This is the clearest evidence for our
thesis: capability tracks pretraining scale, not architecture.

*Caveats (important for fair comparison).* The LLM is **not a level-field
competitor**: it is not trained on our split and has vastly more
pretraining, so it sits in a separate reference tier. Only a single
zero-shot prompt setting was evaluated (no few-shot). Because predictions
were precomputed and cached, **per-call cost and latency were not recorded;
we therefore make no cost-based claims** and treat the LLM purely as an
accuracy reference. MUStARD is public, so contamination (the model having
seen it in pretraining) cannot be ruled out.

**Published MUStARD context.** Prior MUStARD work is **multimodal**
(text + audio + video) and reports higher figures than our text-only
setting; those numbers are discussed as context in the literature review,
not as head-to-head competitors, because the modality and splits differ.
No published Tagalog/Taglish MUStARD baseline exists, which is itself a
gap this work begins to fill.

## 7 Conclusion

We set out to measure the complexity–payoff trade-off in sarcasm detection
and to test cross-lingual transfer between English and Taglish. Three
findings stand out.

First, **complexity did not pay off** in our small, text-only regime: a
TF-IDF baseline matched or beat every from-scratch and fine-tuned neural
model in EN→EN. Second, **transfer behaviour splits by model family** —
from-scratch neural models collapse on EN→TL, while TF-IDF and pretrained
multilingual models degrade gracefully; target-language fine-tuning
recovers most of the loss for the weaker models, though not uniformly
(XLM-R is an instructive exception). Third, **a zero-shot LLM outperformed
all trained models**, most clearly on Taglish, indicating that the decisive
factor is large-scale pretraining rather than architectural sophistication.

The takeaway: on low-resource, small-data sarcasm tasks, architectural
complexity is not the lever it is often assumed to be. A strong classical
baseline, a pretrained multilingual model, or an LLM reference are each
more rational starting points than a bespoke from-scratch Transformer.

## 8 Recommendation

- **For practitioners (short term).** Start with TF-IDF + logistic
  regression as a baseline for Taglish sarcasm; it is cheap, robust, and
  surprisingly hard to beat at this data scale. Reserve neural models for
  when substantially more labelled data is available.
- **For cross-lingual deployment.** Do not reuse a from-scratch
  English-trained model on Taglish — it collapses. Use a pretrained
  multilingual model, and where feasible fine-tune on target-language data,
  which significantly helps most models.
- **For an accuracy ceiling.** A zero-shot LLM is the strongest option,
  especially for code-switched Taglish; treat it as a reference or
  high-value fallback (with the caveat that cost/latency must be measured
  before production use).
- **Future work.** Acquire more labelled Taglish data (the binding
  constraint), measure LLM cost/latency directly, and explore few-shot
  prompting and a published multimodal MUStARD comparison.

## Member Contribution

| Member | Contributions |
|--------|---------------|
| {{NAME 1}} | {{e.g. Data pipeline, EDA, Fleiss' kappa, data section}} |
| {{NAME 2}} | {{e.g. Classical + CNN/BiGRU models, regularization}} |
| {{NAME 3}} | {{e.g. From-scratch attention ladder, ablations}} |
| {{NAME 4}} | {{e.g. Pretrained models, LLM reference, stats, CI/tests}} |

## References

- Castro, S., et al. (2019). *Towards Multimodal Sarcasm Detection (An
  Obviously Perfect Paper).* ACL. (MUStARD dataset)
- Vaswani, A., et al. (2017). *Attention Is All You Need.* NeurIPS.
- Chollet, F., & Watson, M. (2026). *Deep Learning with Python*, 3rd ed.
  Manning.
- Zhang, A., et al. (2023). *Dive into Deep Learning.* Cambridge.
- {{Any HF model cards used: distilbert-base-multilingual-cased,
  bert-base-multilingual-cased, xlm-roberta-base, roberta-tagalog-base}}
- {{LLM used: model name + access date}}
- {{Any other sources, and a note on LLMs used to assist development}}

## Acknowledgements

We thank our three annotators for validating the Tagalog/Taglish
translations, and the ML3 teaching team for guidance.